# 01 · Preprocess: clean, join, split

First notebook of the ESCI Task 1 pipeline (us, small). Runs on **Kaggle CPU** (no GPU, no internet) in a few minutes.

**Input:** the `esci-raw` dataset with 4 files:
- `examples_us_small.parquet`, `products_us_small.parquet`, `sources_us_small.csv` (raw data);
- `query_groups_val_proposal.csv` (from `eda/tables/`): validation split by near-duplicate query groups, seed 42.

**Output** (`/kaggle/working`):

| File | Content |
|---|---|
| `products_clean.parquet` | Cleaned fields, `text_title_bc`, `text_main`, `has_*` flags |
| `pairs.parquet` | `example_id, query_id, query, product_id, esci_label, gain, source, split, fold10` |
| `split_manifest.csv` | `query_id, split, dup_group, fold10, seed` |
| `data_check.json` | Sanity-check numbers |
| `label_conflicts.csv` | Products with the same title in one query but different labels (for error analysis) |

Later notebooks read these files through **Add Input → Notebook Output**. Download the 3 small files (`split_manifest.csv`, `data_check.json`, `label_conflicts.csv`) into `data/splits/` in the repo and push them to GitHub.

**Decisions applied:**
- Light cleaning: HTML removed; case, digits, units and punctuation kept.
- The validation set is exactly `split_eda` from the EDA audit.
- `fold10` is assigned by `dup_group` and balanced by number of pairs: 2 folds = `fold10 // 5`, 5 folds = `fold10 // 2`.

In [1]:
import glob, html, json, re, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N_FOLDS = 10
SEP = ' [SEP] '   # separator between product fields in the text_* columns
GAIN = {'E': 1.0, 'S': 0.1, 'C': 0.01, 'I': 0.0}   # official ESCI gains
OUT = Path('/kaggle/working')


def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, (f'{name} not found under /kaggle/input: add the esci-raw dataset as an input '
                  '(query_groups_val_proposal.csv is in eda/tables/ on your machine; upload it into esci-raw, '
                  'then click Update on the input)')
    return Path(hits[0])


EXAMPLES = find('examples_us_small.parquet')
PRODUCTS = find('products_us_small.parquet')
SOURCES = find('sources_us_small.csv')
VAL_GROUPS = find('query_groups_val_proposal.csv')
print('data from', EXAMPLES.parent, '| val groups from', VAL_GROUPS.parent)

data from /kaggle/input/datasets/dchromevn/esci-raw | val groups from /kaggle/input/datasets/dchromevn/esci-raw


## 1. Load and check the raw data

In [2]:
ex = pd.read_parquet(EXAMPLES)
pr = pd.read_parquet(PRODUCTS)
src = pd.read_csv(SOURCES)
print('examples', ex.shape, '| products', pr.shape, '| sources', src.shape)

assert len(ex) == 601_354 and len(pr) == 482_105
assert ex['example_id'].is_unique and pr['product_id'].is_unique
assert not ex.duplicated(['query_id', 'product_id']).any(), 'duplicate query-product pairs'
assert set(ex['product_id']) <= set(pr['product_id']), 'examples reference unknown products'
assert set(ex['query_id']) <= set(src['query_id']), 'queries without a source'
assert ex.groupby('query_id')['split'].nunique().max() == 1, 'a query is in both train and test'
print('queries per split:', ex.groupby('split')['query_id'].nunique().to_dict())

examples (601354, 9) | products (482105, 7) | sources (29844, 2)
queries per split: {'test': 8956, 'train': 20888}


## 2. Clean the product fields

Light cleaning:
- unescape HTML entities, strip HTML tags, NFC normalisation, collapse whitespace;
- bullet points are newline-separated and are re-joined with `' ; '`;
- **no** lowercasing and no punctuation removal; digits and units (`64GB`, `10.2"`, `w/o`) are kept, because they help tell E from S;
- fields that are empty after cleaning become `None`.

In [3]:
TAG = re.compile(r'<[a-zA-Z/!][^>]{0,200}>')   # '<' followed by a letter, so '< 5' or '<3' survive
WS = re.compile(r'\s+')


def clean_text(s):
    """HTML entities and tags out, NFC, single spaces. Case, digits, units and punctuation are kept."""
    if not isinstance(s, str):
        return None
    s = html.unescape(TAG.sub(' ', html.unescape(s)))   # unescape first so '&lt;br&gt;' is removed too
    s = WS.sub(' ', unicodedata.normalize('NFC', s)).strip()
    return s or None


def clean_bullets(s):
    """Bullet points are newline-separated: clean each one and join them with ' ; '."""
    if not isinstance(s, str):
        return None
    items = [b for b in (clean_text(x) for x in s.split('\n')) if b]
    return ' ; '.join(items) or None


RAW_COLS = {'title': 'product_title', 'brand': 'product_brand', 'color': 'product_color',
            'bullets': 'product_bullet_point', 'description': 'product_description'}

t0 = time.time()
prod = pd.DataFrame({'product_id': pr['product_id']})
for col, raw_col in RAW_COLS.items():
    prod[col] = pr[raw_col].map(clean_bullets if col == 'bullets' else clean_text)
print(f'cleaned {len(prod):,} products in {time.time() - t0:.0f} s')
assert prod['title'].notna().all(), 'a product lost its title'


def missing_rate(s):
    return float((s.isna() | (s.fillna('').str.strip() == '')).mean())


def html_rate(s):
    return float(s.fillna('').str.contains(TAG.pattern, regex=True).mean())


check = pd.DataFrame({
    'missing_raw': {c: missing_rate(pr[r]) for c, r in RAW_COLS.items()},
    'missing_clean': {c: float(prod[c].isna().mean()) for c in RAW_COLS},
    'html_raw': {c: html_rate(pr[r]) for c, r in RAW_COLS.items()},
    'html_clean': {c: html_rate(prod[c]) for c in RAW_COLS},
}).round(4)
# EDA reference: missing description 0.4778, bullets 0.132, brand 0.0564, color 0.3218; HTML in description ~0.29
check

cleaned 482,105 products in 42 s


,missing_raw,missing_clean,html_raw,html_clean
title,0.0000,0.0000,0.0000,0.0
brand,0.0564,0.0564,0.0000,0.0
color,0.3218,0.3218,0.0000,0.0
bullets,0.1320,0.1320,0.0032,0.0
description,0.4778,0.4779,0.2923,0.0


Inspect a few examples before and after cleaning (descriptions with HTML, and bullet points).

In [4]:
has_html = pr['product_description'].fillna('').str.contains(TAG.pattern, regex=True)
for i in pr.index[has_html].to_series().sample(5, random_state=SEED):
    print('RAW  :', pr.at[i, 'product_description'][:300])
    print('CLEAN:', (prod.at[i, 'description'] or '')[:300])
    print()
for i in pr.index[pr['product_bullet_point'].notna()].to_series().sample(2, random_state=SEED):
    print('RAW bullets  :', repr(pr.at[i, 'product_bullet_point'][:200]))
    print('CLEAN bullets:', (prod.at[i, 'bullets'] or '')[:200])
    print()

RAW  : <br>Thank you for shopping at windaze, your satisfaction is our always goal.</b><br><br> This daybed provides a variety of functions. <br> This daybed can be used as a spacious sofa during the day and a comfortable single or king bed at night. <br> This daybed has a Trundle bed provides a widely sle
CLEAN: Thank you for shopping at windaze, your satisfaction is our always goal. This daybed provides a variety of functions. This daybed can be used as a spacious sofa during the day and a comfortable single or king bed at night. This daybed has a Trundle bed provides a widely sleeping platforms for double

RAW  : <p>【Double faced Ribbon】Silky texture with double sided ribbon.Width: 1-1/2" Length: 25yards each roll with plastic spool, can be store easily</p><p>【TRUE COLOR】Colors are exactly as picture shown, machine washable with no color fade, held up well in various weather conditions</p><p>【Plus Saving】Plu
CLEAN: 【Double faced Ribbon】Silky texture with double sided ribbon.Width: 1

## 3. Product text presets

| Preset | Text |
|---|---|
| `title` | title |
| `title_bc` | `title [SEP] brand: X [SEP] color: Y` |
| `main` | `title_bc [SEP] bullets` |
| `all` | `main [SEP] description` |

Missing fields are dropped and the title always comes first, so when a text is longer than `max_len`, truncation cuts the bullets or description, never the title.

`text_title_bc` and `text_main` are stored in `products_clean.parquet`. The `all` preset is only for ablations: copy `product_text` into that notebook when it is needed.

In [5]:
PRESETS = {
    'title': ['title'],
    'title_bc': ['title', 'brand', 'color'],
    'main': ['title', 'brand', 'color', 'bullets'],
    'all': ['title', 'brand', 'color', 'bullets', 'description'],
}
LABEL = {'brand': 'brand: ', 'color': 'color: ', 'bullets': '', 'description': ''}


def product_text(df, fields):
    """Text of each product for a preset; missing fields are dropped and the title always comes first."""
    cols = PRESETS[fields]
    out = df[cols[0]].astype(str)
    for c in cols[1:]:
        v = df[c]
        out = out + (SEP + LABEL[c] + v.fillna('')).where(v.notna(), '')
    return out


prod['text_title_bc'] = product_text(prod, 'title_bc')
prod['text_main'] = product_text(prod, 'main')
for c in ['brand', 'color', 'bullets', 'description']:
    prod[f'has_{c}'] = prod[c].notna()

for t in prod['text_main'].sample(3, random_state=SEED):
    print(t[:400], '\n')
words = prod['text_main'].str.split().str.len()
# EDA (without field labels): median 95 words, p75 174, 20.7% longer than 190 words
print('text_main words: median', int(words.median()), '| p75', int(words.quantile(0.75)),
      '| share > 190 words', round(float((words > 190).mean()), 3))

Ace Tech Cellular LCD Screen Replacement Compatible with iPhone 11 Pro Max 6.5 inch (Model A2161, A2220, A2218) 3D Touch Screen Display Digitizer Frame Assembly Repair Kit with Repair Tools [SEP] brand: Ace Tech Cellular [SEP] COMPATIBLE MODELS - Compatible with iPhone 11 PRO MAX 6.5 inch. Model number on the back cover: A2161, A2220, A2218 ; TUTORIAL & MANUAL – We provide step by step tutorial an 

The Everything Pet Rabbit Handbook: Your Ultimate Guide to Pet Rabbit Ownership, Training, and Care [SEP] brand: Createspace Independent Publishing Platform 

BW Technologies XT-XWHM-Y-NA GasAlertMax XT II 4-Gas Detector with Pump, Combustible, O2, H2S and CO, Yellow [SEP] brand: BW Technologies [SEP] color: Yellow [SEP] Four-gas detector for surveying levels of H2S, oxygen (O2), carbon monoxide (CO), and combustibles in natural and industrial environments, and in other personal safety, training, research, and ambient air monitoring applications ; Elect 

text_main words: median 104 | p75 1

## 4. Split manifest

- **val:** exactly `split_eda` from `query_groups_val_proposal.csv`. Near-duplicate query groups (`dup_group`) are never split between train and val; a group with a query in test stays in train.
- **`fold10`** (0–9, train only): shuffle the train `dup_group`s (seed 42) and give each one to the fold with the fewest pairs so far. Val and test have `fold10 = -1`.

In [6]:
groups = pd.read_csv(VAL_GROUPS)   # query_id, split, split_eda, dup_group (eda/eda_audit.py, seed 42)
q = ex.groupby('query_id').agg(split=('split', 'first'), n_pairs=('product_id', 'size')).reset_index()
q = q.merge(groups[['query_id', 'split', 'split_eda', 'dup_group']].rename(columns={'split': 'split_file'}),
            on='query_id', how='left')
assert q['dup_group'].notna().all(), 'queries missing from query_groups_val_proposal.csv'
assert (q['split'] == q['split_file']).all(), 'official split differs from the EDA file'
q['split'] = q['split_eda']

# fold10: shuffle the train dup_groups, give each one to the fold with the fewest pairs so far
train_groups = q[q['split'] == 'train'].groupby('dup_group')['n_pairs'].sum()
load = np.zeros(N_FOLDS, dtype=int)
fold_of = {}
for g in np.random.default_rng(SEED).permutation(train_groups.index.to_numpy()):
    f = int(np.argmin(load))
    fold_of[g] = f
    load[f] += train_groups[g]
q['fold10'] = np.where(q['split'] == 'train', q['dup_group'].map(fold_of), -1).astype(int)

manifest = q[['query_id', 'split', 'dup_group', 'fold10']].assign(seed=SEED)
counts = manifest['split'].value_counts()
print('queries:', counts.to_dict(), '| pairs per fold:', load.tolist())
assert counts['train'] == 18_799 and counts['val'] == 2_089 and counts['test'] == 8_956
assert (q[q['split'] != 'test'].groupby('dup_group')['split'].nunique() == 1).all(), 'a dup_group is in train and val'
assert (q[q['split'] == 'train'].groupby('dup_group')['fold10'].nunique() == 1).all(), 'a dup_group is in two folds'

queries: {'train': 18799, 'test': 8956, 'val': 2089} | pairs per fold: [37834, 37826, 37809, 37810, 37800, 37808, 37826, 37804, 37818, 37805]


## 5. Query–product pairs and label-distribution check

`pairs.parquet` holds only keys and labels; product text comes from `products_clean.parquet` through a join on `product_id`. `small_version` and `large_version` are not carried over, since they are forbidden features.

In [7]:
pairs = (ex[['example_id', 'query_id', 'query', 'product_id', 'esci_label']]
         .merge(src[['query_id', 'source']], on='query_id', how='left')
         .merge(q[['query_id', 'split', 'fold10']], on='query_id', how='left'))
pairs['query'] = pairs['query'].map(lambda s: unicodedata.normalize('NFC', s).strip())
pairs['gain'] = pairs['esci_label'].map(GAIN)
pairs = pairs.sort_values(['query_id', 'example_id']).reset_index(drop=True)
assert len(pairs) == 601_354 and pairs[['source', 'split', 'gain']].notna().all().all()

# label share of every split and every train fold vs the whole train part
share = pd.crosstab([pairs['split'], pairs['fold10']], pairs['esci_label'], normalize='index')[list('ESCI')]
ref = pairs.loc[pairs['split'] == 'train', 'esci_label'].value_counts(normalize=True)[list('ESCI')]
gap = (share - ref).abs().max(axis=1)
print('max label-share gap to train:', round(float(gap.max()), 4))
assert gap.max() < 0.015, 'a split or fold has a label distribution far from train'
share.round(4).assign(n_pairs=pairs.groupby(['split', 'fold10']).size(),
                      n_queries=pairs.groupby(['split', 'fold10'])['query_id'].nunique())

max label-share gap to train: 0.0094


esci_label         E       S       C       I  n_pairs  n_queries
split fold10                                                    
test  -1      0.4387  0.3498  0.0446  0.1669   181701       8956
train  0      0.4285  0.3532  0.0435  0.1748    37834       1885
       1      0.4360  0.3442  0.0458  0.1741    37826       1861
       2      0.4288  0.3567  0.0449  0.1696    37809       1874
       3      0.4245  0.3576  0.0458  0.1721    37810       1923
       4      0.4423  0.3484  0.0452  0.1641    37800       1889
       5      0.4356  0.3474  0.0432  0.1737    37808       1892
       6      0.4350  0.3467  0.0475  0.1708    37826       1858
       7      0.4316  0.3506  0.0452  0.1727    37804       1882
       8      0.4379  0.3520  0.0473  0.1628    37818       1861
       9      0.4294  0.3591  0.0472  0.1643    37805       1874
val   -1      0.4359  0.3535  0.0449  0.1657    41513       2089

## 5b. Conflicting labels within a query

Products with the same title (lowercased, whitespace collapsed) inside one query are grouped. A group is conflicting when it carries two or more different labels. The EDA found 322 of 1,301 such groups in the official train part; this caps what a title-only model can get right. Labels are not changed: the file is only for error analysis.

In [8]:
p = pairs.merge(prod[['product_id', 'title']], on='product_id')
p['title_norm'] = p['title'].str.lower().str.split().str.join(' ')
g = p.groupby(['query_id', 'title_norm'])['esci_label']
grp = pd.DataFrame({'size': g.size(), 'n_labels': g.nunique()}).reset_index()
grp = grp[grp['size'] > 1].merge(q[['query_id', 'split']], on='query_id')
grp['part'] = np.where(grp['split'] == 'test', 'test', 'train+val')
conflict_stats = {part: {'same_title_groups': int(len(d)), 'conflicting': int((d['n_labels'] > 1).sum())}
                  for part, d in grp.groupby('part')}
print(conflict_stats)   # EDA, official train (= train+val here): 1301 groups, 322 conflicting

conflicts = (p.merge(grp.loc[grp['n_labels'] > 1, ['query_id', 'title_norm']], on=['query_id', 'title_norm'])
             [['query_id', 'query', 'split', 'title_norm', 'product_id', 'esci_label']]
             .sort_values(['query_id', 'title_norm', 'esci_label']).reset_index(drop=True))
print(len(conflicts), 'rows')
conflicts.head(6)


{'test': {'same_title_groups': 645, 'conflicting': 179}, 'train+val': {'same_title_groups': 1301, 'conflicting': 322}}
1336 rows


,query_id,query,split,title_norm,product_id,esci_label
0,18,#10 envelopes without security tint,test,"quality park #10 self-seal security envelopes,...",B08NW4LRQC,E
1,18,#10 envelopes without security tint,test,"quality park #10 self-seal security envelopes,...",B08P2FWYKG,I
2,34,#4 pads without wings,test,"always ultra thin pads for women, size 1 regul...",B01IZ5EYJO,E
3,34,#4 pads without wings,test,"always ultra thin pads for women, size 1 regul...",B001G7QSGQ,I
4,117,''i am here for a good time not a log time'' s...,test,here for a good time not a long time t-shirt,B07P8JCQHD,E
5,117,''i am here for a good time not a log time'' s...,test,here for a good time not a long time t-shirt,B085SRSLQ6,I


## 6. Save outputs

In [9]:
prod.to_parquet(OUT / 'products_clean.parquet', index=False)
pairs.to_parquet(OUT / 'pairs.parquet', index=False)
manifest.to_csv(OUT / 'split_manifest.csv', index=False)
conflicts.to_csv(OUT / 'label_conflicts.csv', index=False)

summary = {
    'created_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime()),
    'seed': SEED, 'n_folds': N_FOLDS, 'sep': SEP, 'pandas': pd.__version__,
    'pairs': len(pairs), 'products': len(prod),
    'queries_per_split': counts.to_dict(),
    'pairs_per_split': pairs['split'].value_counts().to_dict(),
    'pairs_per_fold': pairs[pairs['split'] == 'train']['fold10'].value_counts().sort_index().to_dict(),
    'cleaning': check.to_dict(),
    'max_label_share_gap': round(float(gap.max()), 4),
    'same_title_label_conflicts': conflict_stats,
}
(OUT / 'data_check.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))

for f in sorted(OUT.iterdir()):
    if f.is_file():
        print(f'{f.name:26s} {f.stat().st_size / 1e6:8.1f} MB')

__notebook__.ipynb              0.0 MB
data_check.json                 0.0 MB
label_conflicts.csv             0.2 MB
pairs.parquet                   9.5 MB
products_clean.parquet        539.1 MB
split_manifest.csv              0.7 MB
